In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
import torch
import math

# **self-attention class**

In [4]:
class SelfAttention:

    # to initialize it
    def __init__(self, dimen=4, heads=1, random_w=True):

        self.dimen = dimen
        self.random_w = random_w
        self.heads = heads

        if random_w:
            self.w_query = torch.rand(dimen, dimen//heads).float()
            self.w_key = torch.rand(dimen, dimen//heads).float()
            self.w_value = torch.rand(dimen, dimen//heads).float()
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, static_embeddings):

        if self.random_w == False:
            
            queries = static_embeddings.float()
            keys = static_embeddings.float()
            values = static_embeddings.float()
            
        else:
        
            queries = torch.matmul(static_embeddings.float(), self.w_query)
            keys = torch.matmul(static_embeddings.float(), self.w_key)
            values = torch.matmul(static_embeddings.float(), self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def mask_weights(self, tensor):
        tensor = tensor.float().clone()
        mask = torch.triu(torch.ones_like(tensor), diagonal=1).bool()
        tensor[mask] = float('-inf')
        return tensor

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)
        #print(contextual_embeddings)
        return contextual_embeddings
        

    # get embeddings of different sentences
    def __call__(self, static_embeddings):

        # 1. obtain query, key, value vectors
        queries, keys, values = self.getQKV(static_embeddings)
        print("query, key, value shapes: ", queries.shape, keys.shape, values.shape)

        # 2. perform dot product bw query and key
        dot_product = self.dotproduct(queries,keys)
        print("dp shape: ", dot_product.shape)

        # 3. scale by 1/sqrt(dimen)
        scaled_dp = self.get_sqrt(dot_product)

        # 4. mask the weights
        masked = self.mask_weights(scaled_dp)

        # 5. apply softmax to each row
        weights = self.apply_softmax(masked)

        # 6. multiply w value to get final embeddings
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        
        print("contextual embeddings shape: ", contextual_embeddings.shape)
        
        return contextual_embeddings

In [5]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [6]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [7]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [8]:
model = Word2Vec(sentences, vector_size=4, window=5, min_count=1, sg=1)

In [9]:
input_matrix1 = []

for word in s1:
    embedding = model.wv[word]
    input_matrix1.append(embedding)

np_ip1 = np.array(input_matrix1)
static_embeddings1 = torch.tensor(np_ip1)

#print(query_tensors)

In [10]:
sa_block = SelfAttention(4,True)

In [11]:
ce1 = sa_block(static_embeddings1)

query, key, value shapes:  torch.Size([3, 4]) torch.Size([3, 4]) torch.Size([3, 4])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 4])


In [12]:
input_matrix2 = []

for word in s2:
    embedding = model.wv[word]
    input_matrix2.append(embedding)

np_ip2 = np.array(input_matrix2)
static_embeddings2 = torch.tensor(np_ip2)

In [13]:
ce2 = sa_block(static_embeddings2)

query, key, value shapes:  torch.Size([3, 4]) torch.Size([3, 4]) torch.Size([3, 4])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 4])


In [14]:
print(static_embeddings2[0:2])

tensor([[-0.1254, -0.0941,  0.1845, -0.0383],
        [-0.0134,  0.0059,  0.1276,  0.2252]])


# **multi-head attention**

In [15]:
ambiguous_sentence = "The cat chased the mouse until it stumbled"
ambiguous_sentence = ambiguous_sentence.split(" ")
print(ambiguous_sentence)

['The', 'cat', 'chased', 'the', 'mouse', 'until', 'it', 'stumbled']


In [16]:
def get_embeddings(model, sentence):
    
    embeddings = [model.wv[word] for word in sentence]
    embeddings_np = np.array(embeddings)
    return torch.tensor(embeddings_np)

In [17]:
model = Word2Vec([ambiguous_sentence], vector_size=512, window=5, min_count=1, sg=1)

In [18]:
emb = get_embeddings(model,ambiguous_sentence)
print(emb.shape)

torch.Size([8, 512])


In [19]:
print(emb)

tensor([[-8.3858e-04, -1.2931e-03, -2.7027e-04,  ..., -1.9349e-03,
         -2.9324e-04,  1.3476e-03],
        [ 1.2903e-03, -2.2377e-04,  1.4973e-03,  ...,  1.0317e-04,
         -1.7566e-03,  1.6347e-03],
        [ 1.1490e-03, -5.7839e-04,  6.1753e-04,  ..., -1.6250e-03,
         -2.8566e-05, -5.1714e-04],
        ...,
        [-1.2890e-03,  8.4774e-04, -9.2673e-05,  ...,  1.8470e-03,
         -1.1359e-03,  1.6143e-03],
        [-1.4161e-03, -1.8756e-03, -5.3587e-04,  ..., -5.0609e-04,
          1.4158e-03, -6.7645e-04],
        [-1.0473e-04,  4.6178e-05,  9.9675e-04,  ..., -1.3424e-03,
         -9.7645e-04, -4.4665e-04]])


In [20]:
sa1 = SelfAttention(512,8,True)
sa2 = SelfAttention(512,8,True)
sa3 = SelfAttention(512,8,True)
sa4 = SelfAttention(512,8,True)
sa5 = SelfAttention(512,8,True)
sa6 = SelfAttention(512,8,True)
sa7 = SelfAttention(512,8,True)
sa8 = SelfAttention(512,8,True)

In [21]:
e1 = sa1(emb)
e2 = sa2(emb)
e3 = sa3(emb)
e4 = sa4(emb)
e5 = sa5(emb)
e6 = sa6(emb)
e7 = sa7(emb)
e8 = sa8(emb)

query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64]

In [22]:
print(e1.shape)

torch.Size([8, 64])


In [23]:
print(e2)

tensor([[-1.5515e-03, -5.0810e-03, -1.2684e-02, -7.0220e-03, -7.5259e-03,
          9.3356e-04,  1.6940e-02,  6.4519e-03,  7.7005e-03, -4.7119e-03,
         -1.2860e-02, -5.1003e-03, -4.4998e-03, -5.8383e-03,  9.2847e-03,
         -5.8695e-03,  6.5380e-03,  3.0318e-03,  6.1398e-03, -1.5324e-02,
         -5.1696e-03,  2.2739e-03, -6.0215e-03, -6.5205e-03, -7.2050e-03,
          3.9467e-03, -9.1967e-03,  9.2713e-03,  1.0013e-02,  9.2717e-03,
         -5.5249e-03, -8.2136e-03, -4.5346e-03, -4.8600e-03, -1.6143e-02,
          3.6327e-03, -1.3341e-02, -1.4768e-02, -2.5749e-03,  3.4066e-03,
         -1.3899e-02, -1.3507e-02,  4.3326e-03,  3.1324e-03, -1.3750e-03,
         -5.1092e-03, -8.7315e-03,  4.9137e-03, -6.6904e-03,  4.7534e-03,
          1.7174e-03, -9.4854e-03,  4.2504e-05,  5.3668e-04,  6.0989e-03,
         -4.7990e-03, -9.2413e-04, -4.7322e-03, -5.6794e-03,  7.7557e-03,
         -4.1280e-03, -4.1086e-03, -3.5932e-03, -1.4060e-02],
        [ 4.7435e-03, -1.4265e-03,  7.2488e-03,  5

In [24]:
concatenated = torch.cat([e1,e2,e3,e4,e5,e6,e7,e8], dim=1)
print(concatenated.shape)

torch.Size([8, 512])


In [25]:
print(concatenated)

tensor([[-0.0008, -0.0029, -0.0124,  ..., -0.0081,  0.0012, -0.0029],
        [ 0.0062,  0.0067, -0.0027,  ..., -0.0050, -0.0034,  0.0035],
        [ 0.0017,  0.0023,  0.0004,  ..., -0.0068, -0.0088, -0.0070],
        ...,
        [ 0.0091,  0.0077,  0.0115,  ...,  0.0050,  0.0018,  0.0008],
        [ 0.0067,  0.0065,  0.0088,  ...,  0.0030,  0.0015, -0.0015],
        [ 0.0071,  0.0070,  0.0073,  ...,  0.0034,  0.0001,  0.0010]])


In [26]:
class MultiHeadAttention:
    
    def __init__(self, num_heads=1, dim=4, random_w=True):
        
        self.num_heads = num_heads
        self.dim = dim
        self.random_w = random_w

    def __call__(self, emb):
        
        outputs = []
        
        for i in range(self.num_heads):
            sa = SelfAttention(self.dim, self.num_heads, self.random_w)
            ce = sa(emb)
            outputs.append(ce)

        concatenated = torch.cat(outputs, dim=1)
        return concatenated

In [27]:
ma = MultiHeadAttention(8,512,True)
embb = ma(emb)
print(embb.shape)

query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64]

In [28]:
print(embb)

tensor([[-0.0186, -0.0016,  0.0053,  ..., -0.0031, -0.0008, -0.0068],
        [-0.0021,  0.0041,  0.0077,  ...,  0.0087, -0.0004,  0.0011],
        [-0.0077, -0.0026,  0.0009,  ..., -0.0072, -0.0047,  0.0001],
        ...,
        [ 0.0075,  0.0091,  0.0098,  ...,  0.0028,  0.0081,  0.0092],
        [ 0.0065,  0.0053,  0.0078,  ...,  0.0022,  0.0062,  0.0084],
        [ 0.0060,  0.0063,  0.0073,  ...,  0.0031,  0.0059,  0.0099]])


# **positional encoding**

In [29]:
print(math.sin(1))
print(math.cos(1))

0.8414709848078965
0.5403023058681398


In [30]:
for i in range(2):
    print(i)

0
1


In [31]:
pos = 1
dim = 6

for word in emb:
    pos_enc = []
    for i in range(dim//2): # 1 pair
        pos_enc.append(math.sin(pos/pow(10000,(2*i/dim))))
        pos_enc.append(math.cos(pos/pow(10000,(2*i/dim))))
    print(pos_enc)

[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792

In [32]:
def get_emb(model, word, pos, dim):
        
    emb = model.wv[word]
    
    pos_enc = np.zeros(dim)
    
    for i in range(dim//2): # 1 pair
        pos_enc[2*i] = math.sin(pos/pow(10000,(2*i/dim)))
        pos_enc[2*i+1] = math.cos(pos/pow(10000,(2*i/dim)))

    emb_np = np.array(emb)
    emb_t = torch.tensor(emb_np)

    pos_np = np.array(pos_enc)
    pos_t = torch.tensor(pos_np)

    result = torch.add(emb_t, pos_t)
    return result


def generate_pos_encodings(model,sentence):

    embeddings = [get_emb(model,word,pos,512) for pos,word in enumerate(sentence)]
    return torch.stack(embeddings)

In [33]:
enc = generate_pos_encodings(model,ambiguous_sentence)
print(enc.shape)

torch.Size([8, 512])


In [34]:
apply_ma = MultiHeadAttention(8,512,True)
get_emb = apply_ma(enc)
print(get_emb.shape)

query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64])
query, key, value shapes:  torch.Size([8, 64]) torch.Size([8, 64]) torch.Size([8, 64])
dp shape:  torch.Size([8, 8])
contextual embeddings shape:  torch.Size([8, 64]

# **cross-attention**

In [35]:
class CrossAttention:

    def __init__(self, dimen=4, random_w=True):

        self.dimen = dimen
        self.random_w = random_w

        if random_w:
            self.w_query = torch.rand(dimen, dimen).float()
            self.w_key = torch.rand(dimen, dimen).float()
            self.w_value = torch.rand(dimen, dimen).float()
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, output_emb, input_emb):

        if self.random_w == False:
            
            queries = output_emb.float()
            keys = input_emb.float()
            values = input_emb.float()
            
        else:
        
            queries = torch.matmul(output_emb.float(), self.w_query)
            keys = torch.matmul(input_emb.float(), self.w_key)
            values = torch.matmul(input_emb.float(), self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)
        return contextual_embeddings
        

    def __call__(self, output_emb, input_emb):

        queries, keys, values = self.getQKV(output_emb, input_emb)
        dot_product = self.dotproduct(queries,keys)
        scaled_dp = self.get_sqrt(dot_product)
        weights = self.apply_softmax(scaled_dp)
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        
        return contextual_embeddings

In [36]:
sentences = [["How", "are", "you?"], ["I","am","fine."]]
print(sentences)

[['How', 'are', 'you?'], ['I', 'am', 'fine.']]


In [37]:
def get_emb(model, word, pos, dim):
        
    emb = model.wv[word]
    
    pos_enc = np.zeros(dim)
    
    for i in range(dim//2): # 1 pair
        pos_enc[2*i] = math.sin(pos/pow(10000,(2*i/dim)))
        pos_enc[2*i+1] = math.cos(pos/pow(10000,(2*i/dim)))

    emb_np = np.array(emb)
    emb_t = torch.tensor(emb_np)

    pos_np = np.array(pos_enc)
    pos_t = torch.tensor(pos_np)

    result = torch.add(emb_t, pos_t)
    return result


def generate_pos_encodings(model,sentence):

    embeddings = [get_emb(model,word,pos,4) for pos,word in enumerate(sentence)]
    return torch.stack(embeddings)

In [38]:
model = Word2Vec(sentences, vector_size=4, window=5, min_count=1, sg=1)

In [39]:
input_emb = generate_pos_encodings(model,sentences[0])
output_emb = generate_pos_encodings(model,sentences[1])
print(input_emb.shape)
print(output_emb.shape)

torch.Size([3, 4])
torch.Size([3, 4])


In [40]:
ca = CrossAttention(4,True)
get_emb = ca(output_emb,input_emb)
print(get_emb.shape)

torch.Size([3, 4])


# **layer normalization**

In [41]:
import torch.nn as nn

In [42]:
def apply_layer_norm(embeddings):

    dim = embeddings[0].shape
    layer_norm = nn.LayerNorm(dim)
    norm_embeddings = layer_norm(embeddings)
    return norm_embeddings

In [43]:
embeddings = torch.randn(3,512)
normemb = apply_layer_norm(embeddings)
print(normemb.shape)

torch.Size([3, 512])


# **feedforward neural network**

In [44]:
class FeedForwardNN(nn.Module):

    def __init__(self, dimen):

        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(dimen,2048)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(2048,dimen)

    def forward(self,emb):

        emb = self.fc1(emb)
        emb = self.relu(emb)
        emb = self.fc2(emb)
        return emb

In [45]:
ffnn = FeedForwardNN(512)

In [46]:
output = ffnn(embeddings)
print(output.shape)

torch.Size([3, 512])


# **encoder**

In [47]:
t1 = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
])
t2 = torch.tensor([
    [-1,0,3],
    [8,3,-5],
    [1,4,0]
])
res = t1 + t2
print(res)

tensor([[ 0,  2,  6],
        [12,  8,  1],
        [ 8, 12,  9]])


In [48]:
class Encoder(nn.Module):

    def __init__(self, dimen, heads):

        super(Encoder,self).__init__()

        self.dimen = dimen
        self.mha = MultiHeadAttention(heads,dimen,True)
        self.ffnn = FeedForwardNN(dimen)

    def __call__(self, embeddings):

        attended_emb = self.mha(embeddings)
        print("shape 1: ", embeddings.shape)
        print("shape 2: ", attended_emb.shape)
        add_attended_emb = attended_emb + embeddings
        layer_norm_emb1 = apply_layer_norm(add_attended_emb)
        non_linear_emb = self.ffnn(layer_norm_emb1)
        add_non_linear_emb = non_linear_emb + layer_norm_emb1
        layer_norm_emb2 = apply_layer_norm(add_non_linear_emb)

        return layer_norm_emb2

In [49]:
class MasterEncoder(nn.Module):

    def __init__(self, num_enc, dimen, heads):

        super(MasterEncoder,self).__init__()

        self.dimen = dimen
        self.num_enc = num_enc
        self.heads = heads
        #self.enc = []

        #for i in range(num_enc):
            #new_enc = Encoder(self.dimen,self.heads)
            #self.enc.append(new_enc)

        self.enc = nn.ModuleList([Encoder(dimen,heads) for _ in range(num_enc)])

    def __call__(self, embeddings):

        for enc in self.enc:
            embeddings = enc(embeddings)

        return embeddings

In [50]:
random_embeddings = torch.randn(3,4)

In [51]:
me = MasterEncoder(6,4,2)

In [52]:
final_emb = me(random_embeddings)
print(final_emb.shape)

query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 2])
query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 2])
shape 1:  torch.Size([3, 4])
shape 2:  torch.Size([3, 4])
query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 2])
query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 2])
shape 1:  torch.Size([3, 4])
shape 2:  torch.Size([3, 4])
query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
dp shape:  torch.Size([3, 3])
contextual embeddings shape:  torch.Size([3, 2])
query, key, value shapes:  torch.Size([3, 2]) torch.Size([3, 2]) torc